In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
# Data and model paths
ROOT = Path.cwd().parent
DATA_DIR = ROOT / 'ml' / 'data'
MODELS_DIR = ROOT / 'include' / 'models'

In [3]:
# Model parameters
EPOCHS = 50
LEARNING_RATE = 0.05
BATCH_SIZE = 64

In [4]:
# Load the dataset
train_data = pd.read_csv(DATA_DIR / 'MNIST' / 'pre_train.csv')
test_data = pd.read_csv(DATA_DIR / 'MNIST' / 'test.csv')

print(f'Train data shape: {train_data.shape}')
print(f'Test data shape: {test_data.shape}')

Train data shape: (15000, 50)
Test data shape: (10000, 50)


In [5]:
# Extract features and labels
X_train = train_data.drop('label', axis=1).to_numpy()
X_test = test_data.drop('label', axis=1).to_numpy()

y_train = train_data['label'].to_numpy()
y_test = test_data['label'].to_numpy()

print(f'Train Class distribution:\n{pd.Series(y_train).value_counts().sort_index()}'); print()
print(f'Test Class distribution:\n{pd.Series(y_test).value_counts().sort_index()}')

Train Class distribution:
0    1479
1    1688
2    1498
3    1530
4    1462
5    1353
6    1473
7    1563
8    1463
9    1491
Name: count, dtype: int64

Test Class distribution:
0     986
1    1125
2     999
3    1020
4     975
5     902
6     982
7    1042
8     975
9     994
Name: count, dtype: int64


In [6]:
# Convert data to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [7]:
# Define the NN model
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(49, 16), nn.ReLU(),  # Input layer (7 x 7 = 49 pixels) to hidden layer 1 (16 neurons)
            nn.Linear(16, 10)               # Hidden layer 1 (16 neurons) to output layer (10 classes)
        )

    def forward(self, x):
        return self.network(x)


model = MNISTNet()

In [8]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [9]:
# Training loop
print("Training...")

dataset = torch.utils.data.TensorDataset(X_train, y_train)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

    epoch_loss = total_loss / len(dataset)

    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] Loss: {epoch_loss:.4f}")

Training...
Epoch [ 1/50] Loss: 2.2478
Epoch [ 2/50] Loss: 1.9283
Epoch [ 3/50] Loss: 1.3991
Epoch [ 4/50] Loss: 1.0225
Epoch [ 5/50] Loss: 0.8180
Epoch [ 6/50] Loss: 0.7041
Epoch [ 7/50] Loss: 0.6352
Epoch [ 8/50] Loss: 0.5907
Epoch [ 9/50] Loss: 0.5597
Epoch [10/50] Loss: 0.5371
Epoch [11/50] Loss: 0.5201
Epoch [12/50] Loss: 0.5065
Epoch [13/50] Loss: 0.4953
Epoch [14/50] Loss: 0.4860
Epoch [15/50] Loss: 0.4783
Epoch [16/50] Loss: 0.4706
Epoch [17/50] Loss: 0.4638
Epoch [18/50] Loss: 0.4583
Epoch [19/50] Loss: 0.4532
Epoch [20/50] Loss: 0.4485
Epoch [21/50] Loss: 0.4442
Epoch [22/50] Loss: 0.4386
Epoch [23/50] Loss: 0.4353
Epoch [24/50] Loss: 0.4313
Epoch [25/50] Loss: 0.4278
Epoch [26/50] Loss: 0.4244
Epoch [27/50] Loss: 0.4212
Epoch [28/50] Loss: 0.4176
Epoch [29/50] Loss: 0.4153
Epoch [30/50] Loss: 0.4120
Epoch [31/50] Loss: 0.4088
Epoch [32/50] Loss: 0.4058
Epoch [33/50] Loss: 0.4037
Epoch [34/50] Loss: 0.4006
Epoch [35/50] Loss: 0.3985
Epoch [36/50] Loss: 0.3957
Epoch [37/50] Lo

In [10]:
# Evaluate the model
model.eval()

with torch.no_grad():
    y_pred = model(X_test)
    y_pred_classes = torch.argmax(y_pred, dim=1)

    accuracy = accuracy_score(y_test, y_pred_classes)
    print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 0.8899


In [11]:
class MNISTNetExport(nn.Module):
    def __init__(self):
        super(MNISTNetExport, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(49, 16), nn.ReLU(),
            nn.Linear(16, 10), nn.Softmax(dim=1)  # add Softmax layer to get output probabilities
        )

    def forward(self, x):
        return self.network(x)

export_model = MNISTNetExport()
export_model.load_state_dict(model.state_dict())

export_model.eval()
with torch.no_grad():
    y_pred_export = export_model(X_test)
    y_pred_classes_export = torch.argmax(y_pred_export, dim=1)

    accuracy_export = accuracy_score(y_test, y_pred_classes_export)
    print(f"\nTest Accuracy (Export Model): {accuracy_export:.4f}")


Test Accuracy (Export Model): 0.8899


In [12]:
from aifes import pytorch2aifes

pytorch2aifes.convert_to_fnn_f32_express(export_model, str(MODELS_DIR / 'MNISTNet'))